# Task 1 — Part 1: Steam Data Preparation & Exploration

This notebook is the first real step of the Steam recommendation system project. The goal is to turn the raw Steam files into clean interaction data that later models can use.

We focus on the exact requirements from `project.md` for **Data Preparation & Exploration**:

1. Load and clean the Steam dataset.
2. Explore sparsity, playtime/interaction distribution, long-tail behavior, and temporal patterns.
3. Create a **time-based train/validation/test split**. This is mandatory in the assignment; a random split would leak future behavior into training.
4. Define what counts as a positive interaction.
5. Save processed files that later tasks can reuse for baselines, retrieval, ranking, and the web app.

**Does this task need the UI?** No. The UI is required later in the Demo Webapp section. For Task 1 Part 1, the deliverable is clean data, clear exploration, and a correct chronological split.

## 0. Setup

The raw Steam files are intentionally **not committed to Git** because they are large. Before running this notebook, place these files in:

- `data/raw/steam/steam_reviews.json.gz`
- `data/raw/steam/steam_games.json.gz`

On Google Colab, you can either upload them manually, mount Google Drive, or download them from the UCSD links documented in `data/steam_manifest.md`.

In [ ]:
# If running in a fresh Colab runtime, uncomment the next line.
# !pip install -q pandas numpy matplotlib seaborn pyarrow

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.steam_recsys.data_prep import (
    REVIEWS_PATH,
    GAMES_PATH,
    POSITIVE_HOURS_THRESHOLD,
    MAX_INTERACTIONS,
    build_catalog,
    load_games,
    load_reviews,
    save_outputs,
    subsample_most_recent,
    summarize,
    time_based_split,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

## 1. Check that the raw files exist

A recommender system is only as good as its input data. We start with a basic file check so the notebook fails early with a helpful message if the dataset has not been staged yet.

In [ ]:
required_files = [REVIEWS_PATH, GAMES_PATH]
for path in required_files:
    print(f"{path}: exists={path.exists()} size_mb={(path.stat().st_size / 1024**2):.2f}" if path.exists() else f"{path}: MISSING")

if not all(path.exists() for path in required_files):
    raise FileNotFoundError("Please stage the two raw Steam .json.gz files before continuing.")

## 2. Load and clean the data

The Steam reviews file contains one row per review/play event. For recommendations, we treat each row as an **implicit interaction** between a user and a game.

Cleaning choices:

- `username` becomes `user_id`.
- `product_id` becomes `item_id`.
- `date` becomes `event_time`.
- `hours` is converted to a numeric playtime value.
- Rows missing user, item, date, or playtime are removed.
- If the same user appears with the same game more than once, we keep the latest row so each user-item pair has one final signal.

The metadata file gives us game titles, categories/genres, and store URLs. Later, the web app will use this metadata to display readable game cards.

In [ ]:
games = load_games()
interactions_raw = load_reviews()

print(f"Games metadata rows: {len(games):,}")
print(f"Clean user-item interactions before assignment cap: {len(interactions_raw):,}")

display(games.head())
display(interactions_raw.head())

## 3. Subsample if the dataset is larger than 5 million interactions

The assignment says: if a dataset exceeds 5M interactions, keep the **most recent 5M** and document the strategy.

Why most recent? In a real product, old behavior may not reflect current taste or current game popularity. Keeping recent events also matches our later goal: simulating a live recommender that predicts future behavior from past behavior.

In [ ]:
interactions = subsample_most_recent(interactions_raw, max_rows=MAX_INTERACTIONS)

if len(interactions_raw) > len(interactions):
    print(f"Subsampling applied: kept the most recent {len(interactions):,} interactions out of {len(interactions_raw):,}.")
else:
    print(f"No subsampling needed: {len(interactions):,} interactions is within the assignment cap.")

print("Date range after optional subsampling:", interactions.event_time.min(), "to", interactions.event_time.max())

## 4. Define positive interactions

Steam playtime is implicit feedback: a user does not give a 1–5 star rating here, but their time spent playing is a strong signal.

We define a **positive interaction** as:

> `hours >= 1.0`

This threshold removes extremely tiny play sessions, accidental launches, and near-zero engagement. It is still permissive enough to keep many real signals. We will use this positive flag in baselines and model evaluation later.

In [ ]:
positive_rate = interactions["is_positive"].mean()
print(f"Positive threshold: hours >= {POSITIVE_HOURS_THRESHOLD}")
print(f"Positive interactions: {interactions['is_positive'].sum():,} / {len(interactions):,} ({positive_rate:.2%})")

interactions[["hours", "is_positive"]].describe(percentiles=[.25, .5, .75, .9, .95, .99])

## 5. Build the item catalog

The catalog is the table of games that appear in our interaction data. It joins interaction item IDs to metadata such as title and category.

This matters later because the API and Streamlit UI must show human-readable fields like title, image, and category. We create a Steam CDN header-image URL pattern now so every item has an image candidate for the future UI.

In [ ]:
catalog = build_catalog(interactions, games)
print(f"Catalog items observed in interactions: {len(catalog):,}")
print(f"Items with known metadata category: {(catalog['category'] != 'Unknown').sum():,}")
display(catalog.head())

## 6. Dataset size and sparsity

Sparsity answers this question: out of all possible user-game pairs, how many have an observed interaction?

Recommendation datasets are usually extremely sparse. A user plays only a tiny fraction of all available games, so the model must generalize from very little observed behavior.

In [ ]:
summary = summarize(interactions, catalog)
summary_table = pd.DataFrame([summary]).T.rename(columns={0: "value"})
display(summary_table)

print(f"Sparsity: {summary['sparsity']:.6%}")

## 7. Interaction/playtime distribution

Playtime is usually skewed: many users play a game for a short time, while a small number of users spend hundreds or thousands of hours. We plot log-transformed playtime so the distribution is readable.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(np.log1p(interactions["hours"]), bins=60)
plt.title("Distribution of log(1 + playtime hours)")
plt.xlabel("log(1 + hours)")
plt.ylabel("Number of interactions")
plt.show()

plt.figure(figsize=(8, 4))
sns.countplot(data=interactions, x="is_positive")
plt.title("Positive vs non-positive interactions")
plt.xlabel("Is positive interaction?")
plt.ylabel("Count")
plt.show()

### Interpretation

The histogram tells us whether most interactions are light play sessions or deep engagement. The positive/non-positive count checks whether our 1-hour threshold keeps enough training signal. If nearly everything were positive or nearly everything were negative, the threshold would be less useful.

## 8. Long-tail analysis

A long-tail catalog means a few games get many interactions, while most games get only a few. This is important because popularity baselines usually do well on the head of the catalog but poorly represent niche items.

In [ ]:
item_counts = interactions.groupby("item_id").size().sort_values(ascending=False)
user_counts = interactions.groupby("user_id").size().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(np.arange(1, len(item_counts) + 1), item_counts.values)
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_title("Item long tail")
axes[0].set_xlabel("Item rank by popularity (log)")
axes[0].set_ylabel("Interactions (log)")

axes[1].plot(np.arange(1, len(user_counts) + 1), user_counts.values)
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_title("User activity long tail")
axes[1].set_xlabel("User rank by activity (log)")
axes[1].set_ylabel("Interactions (log)")
plt.show()

popular_items = item_counts.head(10).reset_index(name="interaction_count").merge(catalog, on="item_id", how="left")
display(popular_items[["item_id", "title", "category", "interaction_count"]])

### Interpretation

If the item curve drops quickly, the catalog has a strong long tail. That means catalog coverage will be an important metric later: a recommender that only suggests the top games may get decent recall but provide a boring product experience.

## 9. Temporal patterns

Because our split must be time-based, we need to understand how interactions are distributed over time. This also helps identify strange gaps or bursts in the raw data.

In [ ]:
monthly = interactions.set_index("event_time").resample("ME").size().rename("interactions").reset_index()

plt.figure(figsize=(14, 5))
sns.lineplot(data=monthly, x="event_time", y="interactions")
plt.title("Monthly Steam interactions")
plt.xlabel("Month")
plt.ylabel("Interactions")
plt.show()

display(monthly.tail())

## 10. Mandatory time-based train/validation/test split

We split chronologically:

- **Train**: earliest 72% of rows. Models learn from this period.
- **Validation**: next 8% of rows. We tune hyperparameters here.
- **Test**: latest 20% of rows. We only use this for final evaluation.

The important rule is that validation and test happen **after** training in time. This avoids future data leaking into the model.

In [ ]:
train, validation, test, boundaries = time_based_split(interactions)

split_summary = pd.DataFrame([
    {"split": "train", "rows": len(train), "start": train.event_time.min(), "end": train.event_time.max(), "positive_rate": train.is_positive.mean()},
    {"split": "validation", "rows": len(validation), "start": validation.event_time.min(), "end": validation.event_time.max(), "positive_rate": validation.is_positive.mean()},
    {"split": "test", "rows": len(test), "start": test.event_time.min(), "end": test.event_time.max(), "positive_rate": test.is_positive.mean()},
])

display(split_summary)
print(boundaries)

assert train.event_time.max() <= validation.event_time.min()
assert validation.event_time.max() <= test.event_time.min()

### Interpretation

The assertions above are intentionally simple: they prove the chronological order of the split. This is one of the most important grading points for Task 1 because random splitting would overestimate performance.

## 11. Save processed outputs for later tasks

We save four processed tables:

- `train.parquet`
- `validation.parquet`
- `test.parquet`
- `items.parquet`

These files are ignored by Git because they are generated data, but they are the handoff point for Task 2 baselines, Task 3 retrieval, ranking, and eventually the API/database loader.

In [ ]:
save_outputs(train, validation, test, catalog)

processed_dir = Path("data/processed/steam")
for path in sorted(processed_dir.glob("*.parquet")):
    print(path, f"{path.stat().st_size / 1024**2:.2f} MB")

## 12. What we have completed

For Task 1 Part 1, we now have:

- A cleaned Steam interaction table.
- A documented positive interaction definition: `hours >= 1.0`.
- EDA for sparsity, playtime distribution, long-tail behavior, and temporal patterns.
- A mandatory chronological train/validation/test split.
- Processed files ready for baselines and model training.

The next project step is Task 2: implement Random and Popularity recommenders and evaluate them with Recall@20, NDCG@10, and Catalog Coverage.